In [9]:
!pip install -q \
    langchain-core==1.4.8 \
    langchain-community==0.4.2 \
    langchain-huggingface==1.2.2 \
    langchain-text-splitters==1.1.2 \
    "sentence-transformers>=4,<5" \
    faiss-cpu==1.14.3 \
    pypdf==5.9.0 \
    pymupdf==1.28.0 \
    "transformers>=4.51,<5"

In [10]:
import os, torch
from google.colab import drive

# Seal off the Hugging Face Hub: everything is already local.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# GPU is required for Qwen3. Stop now if it is missing.
assert torch.cuda.is_available(), (
    "No GPU detected. Set: Runtime > Change runtime type > GPU (T4), then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

# Paths: models only (everything else is defined where it is used).
BASE        = "/content/drive/MyDrive/DH2026_Workshop"
ENCODER_DIR = f"{BASE}/Encoder"
LLM_DIR     = f"{BASE}/LLM"

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
checks = {
    "Encoder weights": f"{ENCODER_DIR}/model.safetensors",
    "Encoder pooling": f"{ENCODER_DIR}/1_Pooling/config.json",
    "LLM weights":     f"{LLM_DIR}/model.safetensors",
    "LLM config":      f"{LLM_DIR}/config.json",
}

missing = [name for name, path in checks.items() if not os.path.exists(path)]

for name, path in checks.items():
    mark = "✗" if name in missing else "✓"
    print(f"  {mark}  {name:16s} {path}")

assert not missing, f"Missing models: {missing}"
print("\nModels ready. The rest of the notebook runs offline.")

  ✓  Encoder weights  /content/drive/MyDrive/DH2026_Workshop/Encoder/model.safetensors
  ✓  Encoder pooling  /content/drive/MyDrive/DH2026_Workshop/Encoder/1_Pooling/config.json
  ✓  LLM weights      /content/drive/MyDrive/DH2026_Workshop/LLM/model.safetensors
  ✓  LLM config       /content/drive/MyDrive/DH2026_Workshop/LLM/config.json

Models ready. The rest of the notebook runs offline.


In [12]:
from pypdf import PdfReader

PDF_PATH = f"{BASE}/Source/The-Sustainable-Development-Goals-Report-2025.pdf"

reader   = PdfReader(PDF_PATH)
pages    = [(p.extract_text() or "") for p in reader.pages]
raw_text = "\n".join(pages)

with open("raw_text.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

print(f"{len(pages)} pages, {len(raw_text):,} chars → raw_text.txt")

51 pages, 244,329 chars → raw_text.txt


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter     = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
naive_chunks = splitter.split_text(raw_text)

with open("naive_chunks.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(naive_chunks):
        f.write(f"{'='*70}\n[chunk {i}]  {len(c)} chars\n{'='*70}\n{c}\n\n")

print(f"{len(naive_chunks)} naive chunks → naive_chunks.txt")

535 naive chunks → naive_chunks.txt


In [14]:
BUILD_SCRIPT = f"{BASE}/Code/build_xml.py"

# build_xml.py takes:  [input PDF]  [output XML]
!python "{BUILD_SCRIPT}" "{PDF_PATH}" sdg_report_2025.xml

PDF → XML 변환: /content/drive/MyDrive/DH2026_Workshop/Source/The-Sustainable-Development-Goals-Report-2025.pdf  (51쪽)
Goal 구간: G1=10-11, G2=12-13, G3=14-17, G4=18-19, G5=20-21, G6=22-23, G7=24-25, G8=26-27, G9=28-29, G10=30-31, G11=32-33, G12=34-35, G13=36-37, G14=38-39, G15=40-41, G16=42-43, G17=44-45

  Goal  1 No poverty                               | kp= 4 sub= 6 para= 12
  Goal  2 Zero hunger                              | kp= 4 sub= 6 para= 11
  Goal  3 Good health and well-being               | kp= 4 sub=10 para= 28
  Goal  4 Quality education                        | kp= 3 sub= 7 para= 18
  Goal  5 Gender equality                          | kp= 3 sub= 8 para= 15
  Goal  6 Clean water and sanitation               | kp= 3 sub= 7 para= 14
  Goal  7 Affordable and clean energy              | kp= 3 sub= 6 para= 10
  Goal  8 Decent work and economic growth          | kp= 3 sub= 7 para= 18
  Goal  9 Industry, innovation and infrastructure  | kp= 3 sub= 5 para= 14
  Goal 10 Reduced ine

In [15]:
import json, re, xml.etree.ElementTree as ET

XML_PATH  = f"{BASE}/Source/sdg_report_2025.xml"   # canonical edition in Drive
MAX_WORDS = 172        # heading included — keeps every chunk under 256 tokens


def split_long(text, limit):
    """Too long to embed: cut at sentence boundaries, never mid-sentence."""
    parts, current, count = [], [], 0
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        words = len(sentence.split())
        if current and count + words > limit:
            parts.append(" ".join(current))
            current, count = [], 0
        current.append(sentence)
        count += words
    if current:
        parts.append(" ".join(current))
    return parts


root   = ET.parse(XML_PATH).getroot()
chunks = []

for goal in root.findall("goal"):
    g_num, g_name, g_pages = int(goal.get("number")), goal.get("name"), goal.get("pages")
    seq = 0

    for sub in goal.findall("subsection"):
        heading = sub.get("heading")
        budget  = max(MAX_WORDS - len(heading.split()), 40)

        # 1. paragraphs, pre-split if any single one is oversized
        paragraphs = []
        for p in sub.findall("paragraph"):
            if not p.text:
                continue
            paragraphs += (split_long(p.text, budget)
                           if len(p.text.split()) > budget else [p.text])

        # 2. pack paragraphs into chunks up to the budget
        groups, current, count = [], [], 0
        for para in paragraphs:
            words = len(para.split())
            if current and count + words > budget:
                groups.append(current)
                current, count = [], 0
            current.append(para)
            count += words
        if current:
            groups.append(current)

        # 3. one chunk per group; provenance is identical across the parts
        for i, group in enumerate(groups):
            seq += 1
            metadata = {                           # provenance. never embedded.
                "id":        f"G{g_num:02d}-S{seq:02d}",
                "goal":      g_num,
                "goal_name": g_name,
                "pages":     g_pages,
                "heading":   heading,
            }
            if len(groups) > 1:
                metadata["part"] = f"{i+1}/{len(groups)}"
            chunks.append({
                "metadata": metadata,
                "text": f"{heading}\n" + "\n".join(group),   # only this is embedded
            })

with open("sdg_chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

whole = sum(1 for c in chunks if "part" not in c["metadata"])
print(f"{len(chunks)} chunks → sdg_chunks.jsonl")
print(f"  {whole} subsections fit in one chunk; the rest were split")

176 chunks → sdg_chunks.jsonl
  47 subsections fit in one chunk; the rest were split


In [16]:
from langchain_core.documents import Document

docs = [Document(page_content=c["text"], metadata=c["metadata"]) for c in chunks]

print(docs[0].metadata)

{'id': 'G01-S01', 'goal': 1, 'goal_name': 'No poverty', 'pages': '10-11', 'heading': 'Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach', 'part': '1/2'}


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=ENCODER_DIR,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Encoder loaded from:", ENCODER_DIR)

Encoder loaded from: /content/drive/MyDrive/DH2026_Workshop/Encoder


In [18]:
import json, time

CHUNKS_PATH = f"{BASE}/Source/sdg_chunks.jsonl"     # canonical chunks in Drive

chunks = [json.loads(line) for line in open(CHUNKS_PATH, encoding="utf-8")]

t0 = time.time()
vectors = embeddings.embed_documents([c["text"] for c in chunks])
print(f"{len(vectors)} embeddings done · {time.time()-t0:.1f}s\n")

print(f"dimensions: {len(vectors[0])}")
print(f"first chunk, first 10 values: {[round(x, 4) for x in vectors[0][:10]]}\n")

with open("sdg_embeddings.jsonl", "w", encoding="utf-8") as f:
    for c, v in zip(chunks, vectors):
        f.write(json.dumps({**c, "vector": v}, ensure_ascii=False) + "\n")

print("→ sdg_embeddings.jsonl")

176 embeddings done · 1.1s

dimensions: 384
first chunk, first 10 values: [-0.0698, -0.0433, 0.0213, 0.0764, 0.0087, -0.0009, -0.1072, 0.057, -0.0717, 0.0446]

→ sdg_embeddings.jsonl


In [19]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.vectorstores import FAISS

store = FAISS.from_embeddings(
    text_embeddings=list(zip([c["text"] for c in chunks], vectors)),
    embedding=embeddings,
    metadatas=[c["metadata"] for c in chunks],
    ids=[c["metadata"]["id"] for c in chunks],
)

print(f"{store.index.ntotal} × {store.index.d} dims · {type(store.index).__name__}\n")

i = 0
print("vector store :", store.index.reconstruct(i)[:4].round(4), "...")
print("id table     :", i, "→", store.index_to_docstore_id[i])
print("text store   :", store.docstore.search(store.index_to_docstore_id[i]).metadata["heading"][:50])

176 × 384 dims · IndexFlatL2

vector store : [-0.0698 -0.0433  0.0213  0.0764] ...
id table     : 0 → G01-S01
text store   : Revised poverty estimates show more people in extr


In [20]:
query = "How many people still live in extreme poverty?"

qv = embeddings.embed_query(query)

print(f"query : {query}")
print(f"vector: {len(qv)} dims\n")

print("  dim       value")
print("  ────────────────")
for i in range(10):
    print(f"  {i:3d}   {qv[i]:+.4f}")
print("   ⋮        ⋮")
for i in range(374, 384):
    print(f"  {i:3d}   {qv[i]:+.4f}")

query : How many people still live in extreme poverty?
vector: 384 dims

  dim       value
  ────────────────
    0   +0.0672
    1   -0.0524
    2   -0.0036
    3   +0.0941
    4   -0.0382
    5   -0.0046
    6   -0.0592
    7   +0.0583
    8   -0.0708
    9   +0.0126
   ⋮        ⋮
  374   +0.0087
  375   -0.0921
  376   +0.0398
  377   +0.0101
  378   -0.0512
  379   -0.0629
  380   +0.0663
  381   -0.0827
  382   -0.0374
  383   +0.0296


In [21]:
import numpy as np

my_query = input("Enter your question (English): ")
mv = np.array(embeddings.embed_query(my_query))

print(f"\n{my_query}")
print(f"→ {len(mv)}-dim vector\n")

for r in range(0, 384, 8):
    print(f"  [{r:3d}] " + "  ".join(f"{x:+.4f}" for x in mv[r:r+8]))

Enter your question (English): Africa`s poverty

Africa`s poverty
→ 384-dim vector

  [  0] -0.0415  +0.0828  -0.0541  +0.0947  -0.0328  +0.0165  -0.0283  -0.0061
  [  8] -0.0233  +0.0746  +0.0298  -0.0832  +0.0148  -0.0141  -0.0344  -0.0650
  [ 16] +0.0087  -0.0693  -0.0880  -0.0354  +0.0137  -0.0235  +0.0251  -0.0516
  [ 24] +0.0099  -0.0240  -0.0012  -0.0248  +0.0279  -0.0038  +0.0743  +0.0141
  [ 32] +0.0068  +0.0600  +0.0405  +0.0005  +0.0804  +0.0315  -0.0223  -0.0726
  [ 40] +0.0924  -0.0714  -0.0281  -0.0851  +0.0486  -0.0240  +0.0782  +0.0543
  [ 48] -0.0671  -0.0350  +0.0183  +0.0174  -0.0840  -0.0418  +0.0039  -0.0248
  [ 56] +0.0038  -0.0564  +0.0448  -0.0159  +0.0298  -0.0009  -0.0088  +0.0825
  [ 64] +0.1336  -0.0309  +0.0627  +0.0793  -0.0697  +0.0721  +0.0491  -0.1018
  [ 72] +0.0261  +0.0023  +0.0138  -0.0103  +0.0545  -0.0024  -0.0292  +0.0121
  [ 80] +0.0312  -0.0291  -0.0578  +0.0283  +0.0032  +0.0118  +0.0575  -0.0965
  [ 88] +0.0243  -0.0847  -0.0011  +0.0269  +0.

In [22]:
import numpy as np

query = "How many people still live in extreme poverty?"
qv    = embeddings.embed_query(query)                  # the query as a vector

V = store.index.reconstruct_n(0, store.index.ntotal)   # all chunk vectors
Q = np.array(qv)                                        # the one query vector

print(f"chunk vectors : {len(V)}, each {len(V[0])} numbers")
print(f"query vector  : 1, {len(Q)} numbers\n")

# measure the distance to each chunk
#   (square the differences, sum all 384, take the square root)
distances = []
for v in V:
    diff = v - Q                        # 384 differences
    dist = np.sqrt((diff ** 2).sum())   # square → sum → square root
    distances.append(dist)

print(f"measured {len(distances)} distances, one per chunk.\n")

# sort nearest first
order = np.argsort(distances)

hand_top3 = [store.index_to_docstore_id[int(i)] for i in order[:3]]

for rank, i in enumerate(order[:3], 1):
    print(f"#{rank}  distance {distances[i]:.4f}   {store.index_to_docstore_id[int(i)]}")

chunk vectors : 176, each 384 numbers
query vector  : 1, 384 numbers

measured 176 distances, one per chunk.

#1  distance 0.7861   G01-S01
#2  distance 0.8651   G01-S03
#3  distance 0.8978   G01-S02


In [23]:
query = "How many people still live in extreme poverty?"

results    = store.similarity_search_with_score(query, k=3)
faiss_top3 = [doc.metadata["id"] for doc, _ in results]

print(f"query: {query}\n")
print("STEP 8 (by hand) :", hand_top3)
print("STEP 9 (FAISS)   :", faiss_top3)
print("match            :", hand_top3 == faiss_top3)

query: How many people still live in extreme poverty?

STEP 8 (by hand) : ['G01-S01', 'G01-S03', 'G01-S02']
STEP 9 (FAISS)   : ['G01-S01', 'G01-S03', 'G01-S02']
match            : True


In [24]:
def search_with_sources(q, k=3):
    for rank, (doc, dist) in enumerate(store.similarity_search_with_score(q, k=k), 1):
        m = doc.metadata
        part = f" · part {m['part']}" if "part" in m else ""
        print(f"[{rank}]  distance {dist:.4f}")
        print(f"     {m['id']} · Goal {m['goal']} {m['goal_name']} · pp. {m['pages']}{part}")
        print(f"     {m['heading']}")
        print()
        print(doc.page_content)
        print("─" * 70)

query = "How many people still live in extreme poverty?"
search_with_sources(query)

[1]  distance 0.6179
     G01-S01 · Goal 1 No poverty · pp. 10-11 · part 1/2
     Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach

Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach
The World Bank revised global poverty estimates using updated price data and national poverty lines from over 160 countries in June 2025. The international poverty line was raised from $2.15 (2017 purchasing power parity (PPP)) to $3.00 (2021 PPP). Under the new threshold, 1.5 billion people escaped poverty between 1990 and 2022 – compared to 1.3 billion under the previous line. However, the update leads to an upward revision of extreme poverty. In 2025, an estimated 808 million people will be living in extreme poverty – up from the previous estimate of 677 million – representing 9.9 per cent of the world’s population, or 1 in 10 people. Eradicating extreme poverty by 2030 appears highly unlikely

In [25]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

hf_logging.set_verbosity_error()          # hide generation-flag notices

tokenizer = AutoTokenizer.from_pretrained(LLM_DIR)
model = AutoModelForCausalLM.from_pretrained(
    LLM_DIR,
    dtype="auto",
    device_map="cuda",
)
print("Loaded:", model.config.model_type,
      f"· {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")


class Qwen3LLM(LLM):
    """Qwen3 behind LangChain's LLM interface: call it with llm.invoke(prompt)."""
    max_new_tokens: int = 512

    @property
    def _llm_type(self) -> str:
        return "qwen3-local"

    def _call(self, prompt: str, stop: Optional[List[str]] = None,
              run_manager: Any = None, **kwargs) -> str:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,          # no <think> block, answer only
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,            # greedy: same answer every time
                pad_token_id=tokenizer.eos_token_id,
            )
        answer_ids = out[0][prompt_len:]    # drop the echoed prompt
        return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


llm = Qwen3LLM()
print("Ready.")

Loaded: qwen3 · 596M params
Ready.


In [26]:
import textwrap

query = "How many people still live in extreme poverty?"

answer = llm.invoke(query)
print(textwrap.fill(answer, width=80))

As of the latest data (2023), the number of people living in extreme poverty is
approximately **1.5 billion**. This figure includes individuals who are unable
to meet the basic needs of food, shelter, and healthcare. The situation has been
growing in recent years due to factors such as climate change, economic
inequality, and global crises.


In [27]:
import textwrap

def ask_with_sources(q, k=3):
    # 1. retrieve
    results = store.similarity_search_with_score(q, k=k)

    # 2. build the context from retrieved chunks
    context = "\n\n".join(doc.page_content for doc, _ in results)
    prompt = (
        "Use only the following sources to answer the question. "
        "If the answer is not in the sources, say you cannot find it.\n\n"
        f"{context}\n\n"
        f"Question: {q}"
    )

    # 3. generate
    answer = llm.invoke(prompt)

    # 4. show the answer, then the sources it rests on
    print(textwrap.fill(answer, width=80))
    print("\nSources")
    for doc, _ in results:
        m = doc.metadata
        part = f" · part {m['part']}" if "part" in m else ""
        print(f"  [{m['id']}] Goal {m['goal']} {m['goal_name']} · pp. {m['pages']}{part}")

query = "How many people still live in extreme poverty?"
ask_with_sources(query)

The revised poverty estimates show that in 2025, an estimated 808 million people
will be living in extreme poverty, representing 9.9 percent of the world's
population, or 1 in 10 people.

Sources
  [G01-S01] Goal 1 No poverty · pp. 10-11 · part 1/2
  [G01-S03] Goal 1 No poverty · pp. 10-11
  [G01-S02] Goal 1 No poverty · pp. 10-11 · part 2/2
